# Module 07 AI Workbook — solution and explanation

> Try the exercise yourself before reading this.

## The adversarial bug

[adversarial_fcnn_buggy.py](./adversarial_fcnn_buggy.py) normalises the inputs
with the **batch's own statistics** instead of the model's trained per-feature
values:

```python
mean = X.mean(axis=0)     # batch mean, not the model's fixed mean
std  = X.std(axis=0)      # batch std,  not the model's fixed std
xn   = (X - mean) / std
```

Inference must reproduce training: the network expects each feature normalised by
the **fixed** `mean`/`istd` it was trained with. Using the current batch's
statistics changes the input distribution the network sees, so the outputs are
wrong — even though they keep the right shape `(N,)` and stay in `[0, 1]`, which
is exactly why a shape check or an eyeball misses it. It is also *fast*, so a
"the GPU version is quicker" review passes too.

## The fix

```python
xn  = (X - p["mean"]) * p["istd"]                 # the model's fixed normalisation
h   = xp.maximum(xn @ p["W1"].T + p["b1"], 0.0)   # ReLU hidden layer
y   = h @ p["W2"] + p["b2"]
out = 1.0 / (1.0 + xp.exp(-y))                    # sigmoid output
```

With `xp = cupy` this runs on the GPU and matches the CPU reference within
tolerance, while still being much faster on a large batch. The full version is in
[solutions/fcnn_solution.py](solutions/fcnn_solution.py).

## Why the verification matters

"Faster" and "correct" are independent questions. The reliable test is a
**max-abs-error against a trusted CPU reference** over a large batch; the speedup
number answers the *other* question. [../verify_fcnn.py](./verify_fcnn.py) reports
both — a shape check alone would have passed the buggy version.

## Mapping to the 5-phase loop

- **PREDICT:** from reading the code, flag that normalisation uses batch
  statistics rather than the model's fixed `mean`/`istd`.
- **VERIFY:** compare to a CPU reference (max-abs-error ≤ tol), not just the
  shape, and record the speedup separately.
- **DIAGNOSE:** the GPU version was fast but changed the math; the fix restores
  the trained normalisation and keeps the speedup.


## Run the worked solution

The cells below build and run the solution. Each should finish with `PASS`.

In [ ]:
!python solutions/fcnn_solution.py